In [1]:
import os
import glob
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Set random seed for scientific reproducibility
np.random.seed(42)

def get_image_files(directory):
    """Helper to list all images in a directory efficiently."""
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.tif', '*.tiff')
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(directory, ext)))
        files.extend(glob.glob(os.path.join(directory, ext.upper())))
    return sorted(files)

# Master storage registry
metadata_records = []

# =====================================================================
# 1. DF2023 Train Subset (Strict Limit: 10,000 items)
# =====================================================================
df2023_train_img_dir = "/kaggle/input/datasets/hiwotyirga/the-digital-forensics-2023-dataset-df2023/DF2023_V15_train/COCO_V15"
df2023_train_gt_dir = "/kaggle/input/datasets/hiwotyirga/the-digital-forensics-2023-dataset-df2023/DF2023_V15_train/COCO_V15_GT"

df2023_train_imgs = get_image_files(df2023_train_img_dir)
if df2023_train_imgs:
    train_sample_size = min(10000, len(df2023_train_imgs))
    sampled_train_imgs = np.random.choice(df2023_train_imgs, size=train_sample_size, replace=False)
    
    for img_path in sampled_train_imgs:
        base_name = os.path.basename(img_path)
        gt_path = os.path.join(df2023_train_gt_dir, base_name)
        metadata_records.append({
            'image_path': img_path,
            'mask_path': gt_path if os.path.exists(gt_path) else None,
            'source_dataset': 'DF2023_Train',
            'category': 'manipulation',
            'split': 'train'
        })

# =====================================================================
# 2. DF2023 Validation & Internal Test Subset (Strict Limit: 3,000 items)
# Split evenly: 1,500 validation, 1,500 internal test
# =====================================================================
df2023_val_img_dir = "/kaggle/input/datasets/mubarekahmed/df2023-subset/DF2023_V15_val/COCO_V15"
df2023_val_gt_dir = "/kaggle/input/datasets/mubarekahmed/df2023-subset/DF2023_V15_val/COCO_V15_GT"

df2023_val_imgs = get_image_files(df2023_val_img_dir)
if df2023_val_imgs:
    val_sample_size = min(3000, len(df2023_val_imgs))
    sampled_val_imgs = np.random.choice(df2023_val_imgs, size=val_sample_size, replace=False)
    
    # Split the 3,000 sampled items 50/50 into val and test
    df2023_v_idx, df2023_t_idx = train_test_split(sampled_val_imgs, test_size=0.50, random_state=42)
    
    for img_list, split_label in [(df2023_v_idx, 'val'), (df2023_t_idx, 'test')]:
        for img_path in img_list:
            base_name = os.path.basename(img_path)
            gt_path = os.path.join(df2023_val_gt_dir, base_name) 
            metadata_records.append({
                'image_path': img_path,
                'mask_path': gt_path if os.path.exists(gt_path) else None,
                'source_dataset': 'DF2023_Val',
                'category': 'manipulation',
                'split': split_label
            })

# =====================================================================
# 3. SIDA Forensics Subset (Limit to 6,000 items total -> 70/15/15 Split)
# =====================================================================
sida_paths = {
    'full_synthetic': "/kaggle/input/datasets/mubarekahmed/sida-forensics/full_synthetic",
    'masks': "/kaggle/input/datasets/mubarekahmed/sida-forensics/masks",
    'real': "/kaggle/input/datasets/mubarekahmed/sida-forensics/real",
    'tampered': "/kaggle/input/datasets/mubarekahmed/sida-forensics/tampered"
}

sida_all_records = []
for category, path in sida_paths.items():
    sida_imgs = get_image_files(path)
    for img_path in sida_imgs:
        sida_all_records.append({
            'image_path': img_path,
            'mask_path': None,
            'source_dataset': 'sida-forensics',
            'category': category
        })

# Enforce the global 6,000 limit across SIDA categories evenly via stratify
sida_df_pool = pd.DataFrame(sida_all_records)
if not sida_df_pool.empty:
    # Stratify by category to ensure uniform sampling of full_synthetic, masks, real, tampered
    _, sampled_sida_df = train_test_split(
        sida_df_pool, 
        test_size=min(6000, len(sida_df_pool)), 
        stratify=sida_df_pool['category'], 
        random_state=42
    )
    
    # Apply 70/15/15 Split on the 6,000 sampled items
    sida_train, sida_temp = train_test_split(sampled_sida_df, test_size=0.30, stratify=sampled_sida_df['category'], random_state=42)
    sida_val, sida_test = train_test_split(sida_temp, test_size=0.50, stratify=sida_temp['category'], random_state=42)
    
    sida_train['split'] = 'train'
    sida_val['split'] = 'val'
    sida_test['split'] = 'test'
    
    # Append structured records to master collection
    metadata_records.extend(pd.concat([sida_train, sida_val, sida_test]).to_dict(orient='records'))

# =====================================================================
# 4. So-Fake Setted Combined Subset (Strict Limit: 2k items per split folder)
# =====================================================================
so_fake_paths = {
    'train': "/kaggle/input/datasets/mubarekahmed/so-fake-setted/so_fake_combined/train",
    'validation': "/kaggle/input/datasets/mubarekahmed/so-fake-setted/so_fake_combined/validation",
    'ood_test': "/kaggle/input/datasets/mubarekahmed/so-fake-setted/so_fake_combined/ood_test"
}

for split_name, path in so_fake_paths.items():
    so_fake_imgs = get_image_files(path)
    if so_fake_imgs:
        sample_limit = min(2000, len(so_fake_imgs))
        sampled_so_fake = np.random.choice(so_fake_imgs, size=sample_limit, replace=False)
        assigned_split = 'val' if split_name == 'validation' else ('test' if split_name == 'ood_test' else 'train')
        
        for img_path in sampled_so_fake:
            metadata_records.append({
                'image_path': img_path,
                'mask_path': None,
                'source_dataset': 'so-fake-combined',
                'category': 'synthetic_hybrid',
                'split': assigned_split
            })

# =====================================================================
# Compile and Verify Final Architecture
# =====================================================================
final_metadata_df = pd.DataFrame(metadata_records)
final_metadata_df.to_csv("balanced_dataset_metadata.csv", index=False)

print("\n=== FINAL GENERATED METADATA MATRIX ===")
print(final_metadata_df.groupby(['source_dataset', 'split']).size())


=== FINAL GENERATED METADATA MATRIX ===
source_dataset  split
DF2023_Train    train    10000
DF2023_Val      test      1500
                val       1500
sida-forensics  test       900
                train     4200
                val        900
dtype: int64
